# Explainable ML

A trained, tuned model is only useful if you can trust *why* it predicts what it
does. Explainability (XAI) comes in two flavours:

- **Global** — which features matter to the model *overall*?
- **Local** — why did it make *this* prediction for *this* input?

This mirrors the SHAP/LIME framing Python users know. We'll do one global method
(permutation importance, by hand) and one local method (Kernel SHAP), then peek
at the game theory underneath.

```{note}
To keep the techniques checkable, we explain a **model whose true behaviour we
know**: feature 0 has weight 3, feature 1 weight 1, feature 2 weight 0 (pure
noise). A good explainer should recover exactly that ranking. In practice the
`predict` below would wrap a trained model from an earlier chapter.
```

In [ ]:
:dep rust_shap = { version = "0.1" }
use rust_shap::masked_model::MaskedModel;

// The model: prediction = 3*f0 + 1*f1 + 0*f2. Feature 2 is irrelevant.
struct Model;
impl MaskedModel for Model {
    fn predict(&self, x: &[f64]) -> f64 {
        3.0 * x[0] + 1.0 * x[1] + 0.0 * x[2]
    }
}

// A handful of samples to explain.
let data: Vec<Vec<f64>> = vec![
    vec![1.0, 2.0, 5.0], vec![2.0, 1.0, 3.0], vec![0.5, 3.0, 1.0],
    vec![3.0, 0.5, 4.0], vec![1.5, 2.5, 2.0], vec![2.5, 1.5, 0.0],
];
println!("model + {} samples ready", data.len());

## Global: permutation importance (by hand)

The idea is model-agnostic and only a few lines: **scramble one feature and see
how much the predictions move.** A feature that matters a lot will, when
shuffled, change predictions a lot; an irrelevant feature won't. (We permute a
column by reversing it, so the result is reproducible.)

```{note}
Tree/forest models in `smartcore` also expose a *built-in* feature importance,
which is the cheapest option when you're using them — but permutation importance
works for **any** model, which is why we build it here.
```

In [ ]:
{
    let base_preds: Vec<f64> = data.iter().map(|r| Model.predict(r)).collect();
    let n_features = 3;
    for j in 0..n_features {
        // Reverse column j across the samples = a reproducible permutation.
        let permuted: Vec<f64> = data.iter().rev().map(|r| r[j]).collect();
        let mut total_change = 0.0;
        for (i, row) in data.iter().enumerate() {
            let mut r = row.clone();
            r[j] = permuted[i];
            total_change += (Model.predict(&r) - base_preds[i]).abs();
        }
        println!("feature {} importance = {:.3}", j, total_change / data.len() as f64);
    }
}

Feature 0 ranks highest, feature 1 lower, feature 2 ~0 — the technique recovered
the true importance ordering.

## Local: Kernel SHAP

Permutation importance is global. To explain a *single* prediction, **SHAP**
attributes it across the features so that `base_value + sum(shap) = prediction`.
We use [`rust_shap`](https://docs.rs/rust_shap)'s Kernel SHAP — model-agnostic,
driven by our `predict` and a background dataset:

In [ ]:
use rust_shap::kernel::kernel_shap;

{
    let instance = vec![2.0_f64, 1.0, 5.0];
    let (base, shap) = kernel_shap(&Model, &instance, &data, 200);
    println!("explaining {:?}", instance);
    println!("base value (avg prediction) = {:.3}", base);
    for (j, v) in shap.iter().enumerate() {
        println!("  feature {} contributes {:+.3}", j, v);
    }
    println!("base + sum(shap) = {:.3}  (model says {:.3})",
             base + shap.iter().sum::<f64>(), Model.predict(&instance));
}

Feature 0 carries the largest attribution and feature 2 essentially none — and
the contributions sum back to the prediction, SHAP's defining property.

## Where SHAP comes from: Shapley values

SHAP borrows the **Shapley value** from cooperative game theory: a fair way to
split a payout among players based on their marginal contributions to every
coalition. The [`shapley`](https://docs.rs/shapley) crate computes it on a toy
game — the same math SHAP applies with *features* as the players:

In [ ]:
:dep shapley = { version = "0.1" }
use std::collections::HashMap;
use shapley::{Coalition, Shapley};

{
    // Two players; the value each coalition can achieve together.
    let worth = HashMap::from([
        (Coalition::new(vec![]),      0.0_f64),
        (Coalition::new(vec![1]),    10.0),
        (Coalition::new(vec![2]),    20.0),
        (Coalition::new(vec![1, 2]), 30.0),
    ]);
    let game = Shapley::new(vec![1_u64, 2], worth);
    println!("player 1 fair share = {:.1}", game.shapley_value(1).unwrap());
    println!("player 2 fair share = {:.1}", game.shapley_value(2).unwrap());
}

```{warning}
**Ecosystem maturity — the thinnest corner of this guide.** Rust's explainability
tooling is newer and less battle-tested than Python's SHAP/LIME. `rust_shap` and
`shapley` are early-stage (0.1.x); verify their state before relying on them.
Permutation importance we built **by hand** precisely so it depends on nothing
but your own code. Also note: at least one Rust gradient-boosting crate
(`perpetual`) advertises built-in SHAP/importance, but it currently requires
nightly Rust and so is excluded from this stable image.
```

Next: [Persistence & Deployment](../08-persistence-deployment/saving-and-loading-models.ipynb) —
saving your trained, tuned, explained model and serving predictions from it.